In [11]:
!pip install duckduckgo-search ddgs langchain-experimental

# DuckDuckGo Search tool

In [8]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool = DuckDuckGoSearchRun()
res = search_tool.invoke("Opena Ai launch")
print(res)

OpenAI released Astra on Thursday, its latest AI model and — according to the company — its most powerful and capable one yet. OpenAI claims that Astra represents "a new frontier on computer ... OpenAI says GPT-6.1 Sol delivers significant improvements over GPT-6 Sol across complex professional tasks, including code writing and debugging, document understanding, and executing multistep ... OpenAI on Thursday announced it will begin rolling out its latest artificial intelligence model, GPT-6 Astra, which the company said is the product of "years of research and big bets." OpenAI DevDay 2026 brought Dots, GPT-6.1 Sol, ChatGPT Space, plugin extensions, MCP Events, Codex upgrades and Pro 500. Every announcement, and what it means. OpenAI staged GPT-6 Astra's launch, angering paying subscribers who lacked access. Sam Altman apologized and hoped subscribers could use Astra by the September 5-6 weekend. OpenAI delayed Astra's release to strengthen protections against cyber misuse and unauthor

# Shell Tool

In [13]:
from langchain_community.tools import ShellTool
shell_tool = ShellTool()
res = shell_tool.invoke('ls')
print(res)

Executing command:
 ls
tools.ipynb



# Make our own tool

In [14]:
from langchain_core.tools import tool

In [15]:
# Step 1 - create a function
def multiply(a ,b):
    """Multiply two numbers"""
    return a*b

In [16]:
# Step 2 - add type hints

def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a*b

In [17]:
# Step 3 - add tool decorator

@tool
def multiply(a:int, b:int) -> int:
    "Multiply tow numbers"
    return a*b

In [19]:
res = multiply.invoke({"a":3, "b":5})
print(res)

15


In [22]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply tow numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


# Structured Tool

In [23]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [27]:
class MultiplyInput(BaseModel):
    a: int = Field(description="The first number to multiply")
    b: int = Field(description="The second number to multiply")

In [28]:
def multiply_func(a: int, b:int) -> int:
    return a*b

In [29]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="Multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [33]:
res = multiply_tool.invoke({'a':3, 'b':3})

print(res)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
Multiply
Multiply two numbers
{'a': {'description': 'The first number to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to multiply', 'title': 'B', 'type': 'integer'}}


# Base Tool

In [34]:
from langchain.tools import BaseTool
from typing import Type

In [35]:
class MultiplyInput(BaseModel):
    a: int = Field(description="The first number to multiply")
    b: int = Field(description="The second number to multiply")

In [36]:
class MultiplyTool (BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a:int, b:int) -> int:
        return a*b

In [37]:
multiply_tool = MultiplyTool()

In [38]:
res = multiply_tool.invoke({'a': 3, 'b': 5})

In [39]:
print(res)

15


In [42]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
llm_tools = llm.bind_tools([multiply_tool])

In [44]:
llm_tools.invoke('What is 8 multiplied by 2').tool_calls

[{'name': 'multiply',
  'args': {'a': 8, 'b': 2},
  'id': 'call_XLtWWJzuJ7LdzFn8l0YF2OVP',
  'type': 'tool_call'}]

# Currency Conversion Tool

In [93]:
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency convesrsion factor between a given currency and a target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/82c90d1ad5fc76daba0fc50a/pair/{base_currency}/{target_currency}'
    response = requests.get(url=url)
    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """

    return base_currency_value * conversion_rate

In [94]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency': 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1791158401,
 'time_last_update_utc': 'Mon, 05 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1791244801,
 'time_next_update_utc': 'Tue, 06 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.4346}

In [95]:
# tool binding
llm = ChatOpenAI()
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [96]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage('What is the conversion factor between USD and INR and can you convert 10 usd to inr')]

In [97]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR and can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [98]:
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

In [99]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_ARiroPYHURH618ueTHz1hkqu',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_cwkxBjxORdokJZjKhuREkMs2',
  'type': 'tool_call'}]

In [100]:
import json
for tool_call in ai_message.tool_calls:
    # print(tool_call)
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        messages.append(tool_message1)
    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)


In [101]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR and can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 121, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EVZAHFMkpl9m7Zhp4j9RHCeaQAFUO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a10b5d-9ea8-7ca3-9206-ebd08562a9d1-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'ca

In [103]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is approximately 96.4346. \n\nConverting 10 USD to INR using the conversion factor:\n10 USD = 10 * 96.4346 = 964.346 INR\n\nSo, 10 USD is approximately equal to 964.346 INR.'